## Setup Colab and sync


In [2]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 879 (delta 14), reused 13 (delta 13), pack-reused 854 (from 4)
Receiving objects: 100% (879/879), 159.23 MiB | 62.76 MiB/s, done.
Resolving deltas: 100% (479/479), done.


In [3]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [4]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
absl-py==2.1.0 is already installed.
Installing asttokens==2.4.1...
astunparse==1.6.3 is already installed.
attrs==23.2.0 is already installed.
beautifulsoup4==4.12.3 is already installed.
Installing bert==2.2.0...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for bert: filename=bert-2.2.0-py3-none-any.whl size=3745 sha256=3f4c7460f4673c09711e6a75e96902f1bb895cc0551722233d0c09208878e9fb
  Stored in directory: /root/.cache/pip/wheels/55/82/8d/a9bad0b8280eb858aa3dcb4e617ee5a1653fdeb239e1e8c3fe
  Created wheel for erlastic: filename=erlastic-2.0.0-py3-none-any.whl size=6780 sha256=f73211e5a8abca55895f1d545099f54d8af4828a791d7b2e1afb90a9b8daddbf
  Stored in directory: /root/.cache/pip/wheels/63/ea/24/ab8ff86604f1a87ca69a06af89bb7e080a5

In [5]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## Load and process embeddings and patents with tech field

In [6]:
#@title Functions

import dill as pickle
import pandas as pd


def save(obj, filename):
    with open(filename, 'wb') as f:
        pickle.dump(obj, f)

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


In [7]:
#@title Load

# Path
path_embeddings = r'/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl'
path_patens = r'/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Patents with tech fields.pkl'

embeddings = load(path_embeddings) ## Dictionary where keys are PatentID and values are embeddings
patents = load(path_patens) ## DataFrame

In [9]:
print(len(embeddings))
print(len(patents))

9075421
8092951


In [8]:
#@title Combine
# 3. Convert embeddings dictionary into a pandas Series and merge.
embedding_series = pd.Series(embeddings, name='embedding')

# Merge the embeddings into the metadata DataFrame.
patents = patents.join(embedding_series)
# Optionally, drop patents without an embedding.
patents = patents.dropna(subset=['embedding'])




In [10]:
patents.head(10)

,cpc_subclass,application_year,grant_year,embedding
patent_id,,,,
10000000,G01S,2015,2018,"[-0.2319, -0.5854, -0.3345, 0.1771, 0.462, -0...."
10000001,B29C,2015,2018,"[0.586, -1.353, -0.06964, 0.3408, -0.3367, -0...."
10000002,B29C,2014,2018,"[-0.004852, 0.593, -1.2705, 0.2257, -0.796, 0...."
10000003,B29C,2013,2018,"[-0.257, -0.998, -0.62, 0.8584, 0.4792, -0.139..."
10000004,B29C,2015,2018,"[-0.6353, -1.116, -0.4683, -0.1401, 0.03093, -..."
10000005,B29C,2012,2018,"[-0.4216, -0.11597, -0.6816, 0.6265, 0.1705, -..."
10000006,B29C,2015,2018,"[0.2878, -1.322, -1.132, 1.071, -0.532, -0.923..."
10000007,B29C,2016,2018,"[-0.8057, -0.9014, 0.5444, -0.0222, 0.178, 0.9..."
10000008,B29C,2014,2018,"[-1.5625, -0.4666, -1.394, 1.299, -1.194, 0.38..."


In [18]:
#@title Process and save
import time

root_folder_app = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Field - Year embeddings/Field - Application Year'  # for application_year grouping
root_folder_grant = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Field - Year embeddings/Field - Grant Year'       # for grant_year grouping

total_count_app = 0
total_count_grant = 0

start_time = time.time()
# 4. Group by CPC subclass and application_year, then save.
for (cpc_subclass, application_year), group in patents.groupby(['cpc_subclass', 'application_year']):
    folder_path = os.path.join(root_folder_app, cpc_subclass, str(application_year))
    os.makedirs(folder_path, exist_ok=True)
    # Convert the group to a dictionary with patent_id as keys.
    group_dict = group.to_dict(orient='index')
    file_path = os.path.join(folder_path, 'patents_with_embeddings.pkl')
    with open(file_path, 'wb') as f:
        pickle.dump(group_dict, f)
    #print(f"Saved {len(group)} patents to {file_path}")
    total_count_app += len(group)

    if time.time() - start_time > 60:
        print(f"Total patents saved: {total_count_app}")
        start_time = time.time()

print(f"Total patents saved for applicatoin year: {total_count_app}")

total_count_grant = 0

# 5. Group by CPC subclass and grant_year, then save.
for (cpc_subclass, grant_year), group in patents.groupby(['cpc_subclass', 'grant_year']):
    folder_path = os.path.join(root_folder_grant, cpc_subclass, str(grant_year))
    os.makedirs(folder_path, exist_ok=True)
    group_dict = group.to_dict(orient='index')
    file_path = os.path.join(folder_path, 'patents_with_embeddings.pkl')
    with open(file_path, 'wb') as f:
        pickle.dump(group_dict, f)
    #print(f"Saved {len(group)} patents to {file_path}")

    total_count_grant += len(group)
    if time.time() - start_time > 30:
        print(f"Total patents saved: {total_count_grant}")
        start_time = time.time()

print(f"Total patents saved for grant year: {total_count_grant}")

print("Processing complete.")

Total patents saved: 87590
Total patents saved: 168001
Total patents saved: 281993
Total patents saved: 534705
Total patents saved: 808014
Total patents saved: 1035947
Total patents saved: 1213498
Total patents saved: 1298333
Total patents saved: 1424761
Total patents saved: 1583935
Total patents saved: 1658937
Total patents saved: 1845838
Total patents saved: 1962938
Total patents saved: 2081660
Total patents saved: 2190378
Total patents saved: 2334890
Total patents saved: 2572354
Total patents saved: 2746070
Total patents saved: 2905457
Total patents saved: 3036369
Total patents saved: 3112886
Total patents saved: 3250664
Total patents saved: 3435815
Total patents saved: 3592070
Total patents saved: 3771292
Total patents saved: 3859334
Total patents saved: 4008384
Total patents saved: 4274984
Total patents saved: 4561537
Total patents saved: 4767711
Total patents saved: 5111170
Total patents saved: 5344328
Total patents saved: 5609222
Total patents saved: 5799404
Total patents saved:

In [15]:
print(total_count_app)

8092951


In [22]:
patents['cpc_subclass'].value_counts()

,count
cpc_subclass,
G06F,583020
H01L,340689
H04L,317549
H04N,213031
A61B,187139
...,...
C12L,9
B68F,8
G21J,7
